<a href="https://colab.research.google.com/github/OrbitalArshia/Trial_AI_Driven_Student_Risk_Assessment/blob/main/Student_Risk_AI_ONE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Student Assessment Score Aggregator

This project provides a Python script to aggregate student assessment scores from two different sources (e.g., student self-assessments and teacher assessments) stored in Google Sheets. It calculates individual scores for various categories based on keywords found in textual feedback and then combines these scores with a weighted average to produce a final assessment for each student.

## Table of Contents
- [Project Description](#project-description)
- [Features](#features)
- [Setup](#setup)
- [Google Sheet Structure](#google-sheet-structure)
- [Keyword Configuration](#keyword-configuration)
- [Usage](#usage)
- [Output](#output)

## Project Description
Educational assessments often involve multiple perspectives. This tool streamlines the process of combining qualitative feedback from different stakeholders (like students and teachers) into quantifiable scores. By leveraging natural language processing (specifically keyword matching), it assigns scores across predefined categories and then merges them into a consolidated view, which is then published to a new Google Sheet.

## Features
- **Google Sheets Integration**: Loads data directly from specified Google Sheet URLs.
- **Keyword-Based Scoring**: Analyzes text entries for positive and negative keywords to assign a numerical score (1-5) for each category.
- **Category Scoring**: Scores students across seven key categories: Behavior, Academic Performance, Attendance Issues, Physical Indicators, Emotional Distress, Academic Diligence, and Rule-Boundary Behavior.
- **Weighted Aggregation**: Combines student and teacher scores using a configurable weighting (e.g., 35% student, 65% teacher).
- **Output to Google Sheets**: Creates a new Google Sheet and writes the final aggregated scores.
- **Error Handling**: Includes basic error handling for sheet loading and parsing issues.

## Setup
To run this script, you'll need a Google Colab environment and the `gspread` and `oauth2client` libraries.

1.  **Open in Google Colab**: Upload the notebook to Google Colab.
2.  **Install Libraries**: Ensure you have the necessary libraries installed. The provided notebook includes the installation command:
    ```bash
    !pip install gspread oauth2client
    ```
3.  **Google Authentication**: The script requires authentication to access Google Sheets. When you run the authentication cell, a pop-up window will appear asking you to grant permissions for Google Drive access. Follow the prompts to authenticate your Google account.
    ```python
    from google.colab import auth
    from google.auth import default
    import gspread

    auth.authenticate_user()
    creds, project = default()
    gc = gspread.authorize(creds)
    ```

## Google Sheet Structure
The script expects two input Google Sheets: one for student feedback and one for teacher feedback. Both sheets should have the following columns:
-   `Student ID`: A unique identifier for each student.
-   Category Columns: Columns corresponding to each of the `CATEGORIES` defined in the script (e.g., `Behavior`, `Academic Performance`, etc.). These columns should contain textual feedback.

**Example Sheet Structure:**

| Student ID | Behavior            | Academic Performance | Attendance Issues |
|:-----------|:--------------------|:---------------------|:------------------|
| S001       | respectful, engaged | grades improving     | on time           |
| S002       | disruptive          | failing              | skipping class    |


## Keyword Configuration
The `KEYWORDS` dictionary defines the positive and negative terms for each assessment category. You can customize this dictionary to better suit your specific feedback and assessment criteria.

```python
CATEGORIES = [
    "Behavior", "Academic Performance", "Attendance Issues",
    "Physical Indicators", "Emotional Distress", "Academic Diligence",
    "Rule\u2011Boundary Behavior"
]

KEYWORDS = {
    "Behavior": {
        "negative": ["disruptive", "rude", ...],
        "positive": ["respectful", "calmer", ...]
    },
    # ... other categories
}
```

## Usage

1.  **Define Sheet URLs**: Update `student_sheet_url` and `teacher_sheet_url` variables with the URLs of your respective Google Sheets.

    ```python
    student_sheet_url = "YOUR_STUDENT_SHEET_URL"
    teacher_sheet_url = "YOUR_TEACHER_SHEET_URL"
    ```

2.  **Run All Cells**: Execute all cells in the Colab notebook in order.

3.  **Review Output**: The script will print the URL of the newly created Google Sheet containing the aggregated scores.

## Output
The script generates a new Google Sheet named `Student_Assessment_Scores_with_StudentID` (or a custom name you provide) containing the final weighted scores for each student across all categories. The scores range from 1 (most negative) to 5 (most positive).

**Example Output in Google Sheet:**

| Student ID | Behavior | Academic Performance | Attendance Issues | Physical Indicators | Emotional Distress | Academic Diligence | Rule‑Boundary Behavior |
|:-----------|:---------|:---------------------|:------------------|:--------------------|:-------------------|:-------------------|:-----------------------|
| S001       | 5.0      | 5.0                  | 4.0               | 5.0                 | 5.0                | 5.0                | 3.6                    |
| S002       | 1.0      | 1.0                  | 1.4               | 1.4                 | 1.0                | 1.0                | 2.0                    |
| S003       | 4.6      | 4.0                  | 3.0               | 5.0                 | 5.0                | 5.0                | 3.6                    |
| ...        | ...      | ...                  | ...               | ...                 | ...                | ...                | ...                    |



In [ ]:
import pandas as pd
import re

def load_sheet(sheet_url):
    match = re.search(r'/d/([a-zA-Z0-9_-]+)/edit.*?gid=(\d+)', sheet_url)
    if match:
        spreadsheet_id = match.group(1)
        gid = match.group(2)
        csv_url = f"https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=csv&gid={gid}"
        try:
            df = pd.read_csv(csv_url, sep=',', on_bad_lines='skip')
            # Heuristic check: If the DataFrame has only one column and its name looks like HTML,
            # or if the first few cells contain HTML, it's likely an HTML page was loaded.
            if len(df.columns) == 1 and df.columns[0].startswith('<!DOCTYPE html>'):
                print(f"Warning: Attempted to load CSV from {csv_url}, but received HTML content. Check sheet permissions or URL format.")
                return pd.DataFrame()
            if not df.empty and df.iloc[0, 0].startswith('<!DOCTYPE html>'):
                 print(f"Warning: Attempted to load CSV from {csv_url}, but received HTML content. Check sheet permissions or URL format.")
                 return pd.DataFrame()

            # Clean column names by stripping whitespace
            df.columns = df.columns.str.strip()
            return df
        except Exception as e:
            print(f"Error loading sheet from {csv_url}: {e}")
            return pd.DataFrame()
    else:
        print(f"Could not parse sheet URL: {sheet_url}")
        return pd.DataFrame()

student_sheet_url = "https://docs.google.com/spreadsheets/d/1BY6vHj0uuf3St0yoD98ZKWyh7gAm0dh9OpviB8gnfko/edit?gid=0#gid=0"
teacher_sheet_url = "https://docs.google.com/spreadsheets/d/12l45_Ve_fXEc-lzU9MefBu5Ibn-KoN4f8y3nE_mMyts/edit?gid=0#gid=0"

students_df = load_sheet(student_sheet_url)
teachers_df = load_sheet(teacher_sheet_url)

students_df.head(), teachers_df.head()

CATEGORIES = [
    "Behavior",
    "Academic Performance",
    "Attendance Issues",
    "Physical Indicators",
    "Emotional Distress",
    "Academic Diligence",
    "Rule\u2011Boundary Behavior" # Changed to use non-breaking hyphen
]

KEYWORDS = {
    "Behavior": {
        "negative": ["disruptive", "rude", "arguing", "acting out", "not listening", "disruptive", "rude", "arguing", "off task", "distracted", "defiant", "interrupting"],
        "positive": ["respectful", "calmer", "improved behavior", "repectful", "calm", "cooperative", "engaged", "mature"]
    },
    "Academic Performance": {
        "negative": ["failing", "grades dropped", "missing assignments", "failing", "struggling", "incomplete", "declining", "missing work"],
        "positive": ["grades improving", "doing better", "passing", "inproving", "progressing", "consistant", "strong effort"]
    },
    "Attendance Issues": {
        "negative": ["skipping class", "absent", "late often","tardy", "skipping", "inconsistant"],
        "positive": ["attendance improved", "on time","punctual","consistant", "inproved attendence"]
    },
    "Physical Indicators": {
        "negative": ["sleeping in class", "very tired", "bags under eyes", "tierd", "drowsy", "sluggish", "restless", "unwell"],
        "positive": ["alert", "energetic", "alert", "healthy"]
    },
    "Emotional Distress": {
        "negative": ["stressed", "anxious", "angry", "withdrawn", "anxious", "angry", "withdrawn", "overwhelmed"],
        "positive": ["confident", "calm", "positive attitude", "confident", "regulated", "positive"]
    },
    "Academic Diligence": {
        "negative": ["unfocused", "not trying", "gives up", "unfocused", "unmotivated", "avoids working", "avoids work", "rushing"],
        "positive": ["studying more", "working hard", "focused", "hardworking", "persistent", "motivated"]
    },
    "Rule\u2011Boundary Behavior": { # Changed to use non-breaking hyphen
        "negative": ["breaking rules", "ignoring instructions", "skipping class", "defiant", "rule-breaking", "ignores instructions", "ignoring instructions", "testing limits"],
        "positive": ["follows rules", "compliant", "responsible",]
    }
}

def score_text(text):
    text = text.lower()
    scores = {cat: 3 for cat in CATEGORIES}
    explanations = {cat: [] for cat in CATEGORIES}

    for category, signals in KEYWORDS.items():
        for phrase in signals["negative"]:
            if phrase in text:
                scores[category] -= 1
                explanations[category].append(f"Negative: '{phrase}'")

        for phrase in signals["positive"]:
            if phrase in text:
                scores[category] += 1
                explanations[category].append(f"Positive: '{phrase}'")

        scores[category] = max(1, min(5, scores[category]))

    return scores, explanations

def score_dataframe(df):
    all_student_category_scores = []

    for index, row in df.iterrows():
        student_id = row['Student ID'] # Changed 'Student_ID' to 'Student ID'
        student_category_scores = {'Student ID': student_id}
        for category in CATEGORIES:
            if category in df.columns:
                text_entry = row[category]
                if pd.notna(text_entry) and isinstance(text_entry, str):
                    # score_text returns all category scores for a given text
                    # We only care about the score for the current category derived from its own text column
                    scores_from_this_text, _ = score_text(text_entry)
                    student_category_scores[category] = scores_from_this_text.get(category, 3)
                else:
                    student_category_scores[category] = 3 # Neutral score for missing/invalid text
            else:
                # If the category column is missing in the DataFrame, assign a neutral score
                student_category_scores[category] = 3.0
                # print(f"Warning: Category column '{category}' not found for Student ID {student_id} in DataFrame. Assigned neutral score (3.0).")
        all_student_category_scores.append(student_category_scores)

    return pd.DataFrame(all_student_category_scores).set_index('Student ID')


student_individual_scores = score_dataframe(students_df)
teacher_individual_scores = score_dataframe(teachers_df)

# Merge the student and teacher scores on 'Student ID'
combined_individual_scores = pd.merge(
    student_individual_scores.add_suffix('_student'),
    teacher_individual_scores.add_suffix('_teacher'),
    on='Student ID',
    how='outer' # Use outer join to keep all students from both dataframes
)

# Initialize a DataFrame for final weighted scores
final_individual_scores = pd.DataFrame(index=combined_individual_scores.index)

# Calculate the final weighted score for each category for each student
for cat in CATEGORIES:
    student_col = f'{cat}_student'
    teacher_col = f'{cat}_teacher'

    # Handle cases where a category might be missing in one of the original dataframes,
    # which would result in NaN after outer join if not explicitly handled by score_dataframe.
    # score_dataframe already assigns 3.0 for missing category columns, so NaNs should indicate
    # a student was missing from one of the DFs entirely, or a dataframe was empty.
    # For simplicity, we'll fill any remaining NaNs with 3.0 before weighting.
    student_scores_filled = combined_individual_scores[student_col].fillna(3.0)
    teacher_scores_filled = combined_individual_scores[teacher_col].fillna(3.0)

    final_individual_scores[cat] = (
        0.35 * student_scores_filled +
        0.65 * teacher_scores_filled
    ).round(1)

final_individual_scores

In [ ]:
print("Columns in students_df:", students_df.columns.tolist())
print("Columns in teachers_df:", teachers_df.columns.tolist())

In [ ]:
# Install the gspread library if you haven't already
# !pip install gspread oauth2client

import gspread
from google.colab import auth
from google.auth import default

# Authenticate to Google Drive. This will prompt you to authorize access.
auth.authenticate_user()

# Authorize gspread client using the authenticated user's credentials
creds, project = default()
gc = gspread.authorize(creds)

# Create a new spreadsheet
# You can give it a meaningful name
spreadsheet_name = 'Student_Assessment_Scores_with_StudentID'
sh = gc.create(spreadsheet_name)

# Share the spreadsheet so others can view it (optional, but requested by user)
# Set permission to 'reader' for 'anyoneWithLink'
sh.share(None, perm_type='anyone', role='reader')

print(f"Spreadsheet '{spreadsheet_name}' created successfully. Link: {sh.url}")

# Get the first worksheet
worksheet = sh.get_worksheet(0)

# Convert the DataFrame to a list of lists, including the header and the 'Student ID' as the first column
data_to_write = [final_individual_scores.reset_index().columns.tolist()] + final_individual_scores.reset_index().values.tolist()

# Update the worksheet with the data
# This will clear existing content and write the new data
worksheet.update('A1', data_to_write)

print("Data written to the Google Sheet successfully!")